# Notebook 06: ResNet50 Baseline Training

## Objective

Train a pretrained ResNet50 model for multi-class pneumonia classification using the processed pediatric chest X-ray dataset.

## Workflow

1. Import Libraries
2. Configuration
3. Dataset Loading
4. Model Initialization
5. Training
6. Validation
7. Testing
8. Performance Evaluation
9. Model Saving

## Step 1: Import Required Libraries

## Import Required Libraries

In [3]:
# ============================================================
# Cell 1: Import Required Libraries
# Notebook 06 - ResNet50 Baseline Training
# ============================================================

# Standard Library
import copy
import random
import time
from pathlib import Path

# Data Processing
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Progress Bar
from tqdm.auto import tqdm

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Pretrained Model
from torchvision.models import (
    resnet50,
    ResNet50_Weights
)

# Evaluation Metrics
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sklearn.utils.class_weight import compute_class_weight

# Plot Style
sns.set_theme(
    style="whitegrid",
    context="notebook"
)

print("=" * 70)
print("Notebook 06 : ResNet50 Baseline Training")
print("Cell 1 : Required Libraries")
print(f"PyTorch Version : {torch.__version__}")
print("=" * 70)

Notebook 06 : ResNet50 Baseline Training
Cell 1 : Required Libraries
PyTorch Version : 2.13.0+cu126


In [4]:
# ============================================================
# Cell 2: Device Configuration
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("Cell 2 : Device Configuration")
print("-" * 70)

print(f"Selected Device : {DEVICE}")

if torch.cuda.is_available():

    print(f"GPU Name     : {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version : {torch.version.cuda}")

    gpu_memory = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024 ** 3)
    )

    print(f"GPU Memory   : {gpu_memory:.2f} GB")

else:
    print("Running on CPU")

print("=" * 70)

Cell 2 : Device Configuration
----------------------------------------------------------------------
Selected Device : cuda
GPU Name     : NVIDIA GeForce RTX 3050
CUDA Version : 12.6
GPU Memory   : 8.00 GB


In [5]:
# ============================================================
# Cell 3: Random Seed Configuration
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("=" * 70)
print("Cell 3 : Random Seed Configuration")
print("-" * 70)
print(f"Random Seed : {SEED}")
print("Reproducibility : Enabled")
print("=" * 70)

Cell 3 : Random Seed Configuration
----------------------------------------------------------------------
Random Seed : 42
Reproducibility : Enabled


In [6]:
# ============================================================
# Cell 4: Hyperparameter Configuration
# ============================================================

# Experiment Information

MODEL_NAME = "ResNet50"
MODEL_ID = "resnet50"

MODEL_VERSION = "Baseline"

EXPERIMENT_NAME = "ResNet50_Baseline"

# Hyperparameters

IMAGE_SIZE = 224

BATCH_SIZE = 16

EPOCHS = 30

LEARNING_RATE = 1e-4

NUM_CLASSES = 3

NUM_WORKERS = 0

CLASS_NAMES = [
    "BACTERIA",
    "NORMAL",
    "VIRUS"
]

print("=" * 70)
print("Cell 4 : Hyperparameter Configuration")
print("-" * 70)

print(f"Model Name      : {MODEL_NAME}")
print(f"Image Size      : {IMAGE_SIZE}")
print(f"Batch Size      : {BATCH_SIZE}")
print(f"Epochs          : {EPOCHS}")
print(f"Learning Rate   : {LEARNING_RATE}")
print(f"Number Classes  : {NUM_CLASSES}")

print("=" * 70)

Cell 4 : Hyperparameter Configuration
----------------------------------------------------------------------
Model Name      : ResNet50
Image Size      : 224
Batch Size      : 16
Epochs          : 30
Learning Rate   : 0.0001
Number Classes  : 3


In [7]:
# ============================================================
# Cell 5: Project Directory Configuration
# ============================================================

# Project Directories

PROJECT_ROOT = Path(
    "/mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI"
)

DATASET_DIR = PROJECT_ROOT / "dataset" / "processed_dataset"

MODELS_DIR = PROJECT_ROOT / "models"

RESULTS_DIR = PROJECT_ROOT / "results"

FIGURES_DIR = PROJECT_ROOT / "figures"

LOGS_DIR = PROJECT_ROOT / "logs"

# Create Directories

for directory in [
    MODELS_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    LOGS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

# Output Files

BEST_MODEL_PATH = MODELS_DIR / f"{MODEL_ID}_best.pth"

CHECKPOINT_PATH = MODELS_DIR / f"{MODEL_ID}_checkpoint.pth"

HISTORY_PATH = RESULTS_DIR / f"{MODEL_ID}_training_history.csv"

CLASSIFICATION_REPORT_PATH = (
    RESULTS_DIR / f"{MODEL_ID}_classification_report.txt"
)

CONFUSION_MATRIX_PATH = (
    FIGURES_DIR / f"{MODEL_ID}_confusion_matrix.png"
)

LOSS_CURVE_PATH = (
    FIGURES_DIR / f"{MODEL_ID}_loss_curve.png"
)

ACCURACY_CURVE_PATH = (
    FIGURES_DIR / f"{MODEL_ID}_accuracy_curve.png"
)

print("=" * 70)
print("Cell 5 : Project Directory Configuration")
print("-" * 70)

print(f"Project Root : {PROJECT_ROOT}")
print(f"Dataset      : {DATASET_DIR}")
print(f"Models       : {MODELS_DIR}")
print(f"Results      : {RESULTS_DIR}")
print(f"Figures      : {FIGURES_DIR}")
print(f"Logs         : {LOGS_DIR}")

print("=" * 70)

Cell 5 : Project Directory Configuration
----------------------------------------------------------------------
Project Root : /mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI
Dataset      : /mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI/dataset/processed_dataset
Models       : /mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI/models
Results      : /mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI/results
Figures      : /mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI/figures
Logs         : /mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI/logs


## Data Preparation

In [8]:
# ============================================================
# Cell 6: Data Augmentation & Image Preprocessing
# ============================================================

# Training Data Transformation
train_transform = transforms.Compose([

    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(degrees=10),

    transforms.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05),
        scale=(0.95, 1.05)
    ),

    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )

])

# Validation Transformation
valid_transform = transforms.Compose([

    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )

])

# Test Transformation
test_transform = valid_transform

print("=" * 70)
print("Cell 6 : Data Augmentation & Image Preprocessing")
print("-" * 70)
print(f"Training Image Size : {IMAGE_SIZE} x {IMAGE_SIZE}")
print("Training Transform  : Ready")
print("Validation Transform: Ready")
print("Test Transform      : Ready")
print("=" * 70)

Cell 6 : Data Augmentation & Image Preprocessing
----------------------------------------------------------------------
Training Image Size : 224 x 224
Training Transform  : Ready
Validation Transform: Ready
Test Transform      : Ready


In [9]:
# ============================================================
# Cell 7: Dataset Loading
# ============================================================

# -------------------------------
# Load Datasets
# -------------------------------

train_dataset = datasets.ImageFolder(
    root=DATASET_DIR / "train",
    transform=train_transform
)

valid_dataset = datasets.ImageFolder(
    root=DATASET_DIR / "validation",
    transform=valid_transform
)

test_dataset = datasets.ImageFolder(
    root=DATASET_DIR / "test",
    transform=test_transform
)

# -------------------------------
# Dataset Information
# -------------------------------

print("=" * 70)
print("Cell 7 : Dataset Loading")
print("-" * 70)

print(f"Training Images   : {len(train_dataset)}")
print(f"Validation Images : {len(valid_dataset)}")
print(f"Test Images       : {len(test_dataset)}")

print("-" * 70)

print(f"Classes           : {train_dataset.classes}")
print(f"Number of Classes : {len(train_dataset.classes)}")

print("-" * 70)

print("Class to Index Mapping:")

for class_name, class_index in train_dataset.class_to_idx.items():
    print(f"  {class_name:<10} -> {class_index}")

print("=" * 70)

Cell 7 : Dataset Loading
----------------------------------------------------------------------
Training Images   : 4099
Validation Images : 878
Test Images       : 879
----------------------------------------------------------------------
Classes           : ['BACTERIA', 'NORMAL', 'VIRUS']
Number of Classes : 3
----------------------------------------------------------------------
Class to Index Mapping:
  BACTERIA   -> 0
  NORMAL     -> 1
  VIRUS      -> 2


## Data Loading

In [10]:
# ============================================================
# Cell 8: DataLoader Creation
# ============================================================

# Enable pin_memory only when using CUDA
PIN_MEMORY = torch.cuda.is_available()

# -------------------------------
# Training DataLoader
# -------------------------------

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False
)

# -------------------------------
# Validation DataLoader
# -------------------------------

valid_loader = DataLoader(
    dataset=valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False
)

# -------------------------------
# Test DataLoader
# -------------------------------

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False
)

# -------------------------------
# Display Information
# -------------------------------

print("=" * 70)
print("Cell 8 : DataLoader Creation")
print("-" * 70)

print(f"Training Batches   : {len(train_loader)}")
print(f"Validation Batches : {len(valid_loader)}")
print(f"Test Batches       : {len(test_loader)}")

print("-" * 70)

print(f"Batch Size         : {BATCH_SIZE}")
print(f"Number of Workers  : {NUM_WORKERS}")
print(f"Pin Memory         : {PIN_MEMORY}")

print("=" * 70)

Cell 8 : DataLoader Creation
----------------------------------------------------------------------
Training Batches   : 257
Validation Batches : 55
Test Batches       : 55
----------------------------------------------------------------------
Batch Size         : 16
Number of Workers  : 0
Pin Memory         : True


In [11]:
# ============================================================
# Cell 8: DataLoader Creation
# ============================================================

# Enable pin_memory only when using CUDA
PIN_MEMORY = DEVICE.type == "cuda"

# -------------------------------
# Training DataLoader
# -------------------------------

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False
)

# -------------------------------
# Validation DataLoader
# -------------------------------

valid_loader = DataLoader(
    dataset=valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False
)

# -------------------------------
# Test DataLoader
# -------------------------------

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False
)

# -------------------------------
# Display Information
# -------------------------------

print("=" * 70)
print("Cell 8 : DataLoader Creation")
print("-" * 70)

print(f"Training Batches   : {len(train_loader)}")
print(f"Validation Batches : {len(valid_loader)}")
print(f"Test Batches       : {len(test_loader)}")

print("-" * 70)

print(f"Batch Size         : {BATCH_SIZE}")
print(f"Number of Workers  : {NUM_WORKERS}")
print(f"Pin Memory         : {PIN_MEMORY}")

print("=" * 70)

Cell 8 : DataLoader Creation
----------------------------------------------------------------------
Training Batches   : 257
Validation Batches : 55
Test Batches       : 55
----------------------------------------------------------------------
Batch Size         : 16
Number of Workers  : 0
Pin Memory         : True
